In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from torch.nn import *
from functools import partial

torch.set_default_tensor_type(torch.cuda.FloatTensor)


class F(Module):
    def __init__(s, f, a=None):
        super().__init__()
        s.f = f
        s.a = a

    def forward(s, x):
        return s.f(x) if s.a is None else s.f(x, s.a)


class Enc(Sequential):
    def forward(s, x):
        while x.size(2) > 1:
            x = super().forward(x)
        return x[...,0]


class EncDec(Sequential):
    def forward(s, x):
        Y = []
        for i in range(int(np.log2(x.size(2)))):
            x = s[0](x)
            y = s[1](x)
            for i in range(i + 1):
                y = s[2](y)
            Y.append(y)
        return torch.cat(Y)


class Add(Sequential):
    def forward(s, x):
        return sum((s(x) for s in s[1:]), s[0](x))


class Res(Sequential):
    def forward(s, x):
        return super().forward(x) + x


class Rep(Sequential):
    def __init__(s, n, *a):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n):
            x = super().forward(x)
        return x


class Mean(Add):
    def forward(s, x):
        return super().forward(x) / len(s)


class Log:
    a = None
    n = 0
    def __init__(s, m=10):
        s.m = m
    def __call__(s, *x):
        if s.a is None:
            s.a = [np.empty((2000,1-hasattr(x,'__iter__')or len(x)), 'f') for x in x]
        for a, x in zip(s.a, x):
            a[s.n] = x
        s.n += 1
        if not s.n % s.m:
            for i, a in enumerate(s.a, 1):
                plt.subplot(len(s.a), 1, i).plot(a[:s.n])
            clear_output(1)
            plt.show()

In [ ]:
from tqdm.notebook import tqdm
a = torch.tensor(np.memmap("inews.7622", "h")).unfold(0, 5, 1)
K = 16
m = Sequential(
    Embedding(7622, K),
    F(partial(torch.transpose, dim0=1, dim1=2)),
    Enc(
#         Add(
#             Sequential(
#                 Conv1d(K, K, 3, padding=1, bias=0),
#                 BatchNorm1d(K, track_running_stats=0),
#                 SiLU(1),
#             ),
#             Conv1d(K, K, 3, padding=1, bias=0),
#         ),
        Add(
            Sequential(
                Conv1d(K, K, 3, padding=1, bias=0),
                BatchNorm1d(K, track_running_stats=0),
                SiLU(1),
#                 Conv1d(K, K, 1, bias=0),
            ),
            Conv1d(K, K, 3, padding=1, bias=0),
        ),
        MaxPool1d(2),
    ),    
    Linear(K, K),
    SiLU(1),
    Linear(K, 7622),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .01, len(a)*20//512, pct_start=.01, div_factor=9, final_div_factor=99)
for i in tqdm(range(O.total_steps // 1000)):
    for t in range(1000):
        y = a[torch.randint(len(a),(512,))].long()
        x = m(y[:,:-1])
        y = y[:,-1]
        t = functional.cross_entropy(x, y)
        t.backward()
        o.step()
        o.zero_grad()
        O.step()
    print(t, (x.argmax(1)==y).sum()/len(y))

In [ ]:
a = np.memmap("inews.7622", "h")

In [ ]:
a,b = np.unique(np.lib.stride_tricks.as_strided(a[:0].view('V5'),[len(a)-4],[2]),return_counts=1)
c,d = np.unique(a.view('V4,V1')['f0'],return_inverse=1)
c = np.zeros(len(c), 'H')
lnp.maximum.at(e, c, b)
e.sum()/d.sum()

In [ ]:
%reset -f in out

In [ ]:
d.shape

In [ ]:
d

In [ ]:
d

In [ ]:
d[c]

In [ ]:
np.log2(t) - m @ np.log2(m) / t

In [ ]:
from tqdm.notebook import tqdm
a = torch.tensor(np.memmap("inews.7622", "h")).unfold(0, 5, 1)
K = 512
# m = Sequential(
#     Embedding(7622, K),
#     F(partial(torch.transpose, dim0=1, dim1=2)),
#     Enc(
# #         Add(
# #             Sequential(
# #                 Conv1d(K, K, 3, padding=1, bias=0),
# #                 BatchNorm1d(K, track_running_stats=0),
# #                 SiLU(1),
# #             ),
# #             Conv1d(K, K, 3, padding=1, bias=0),
# #         ),
#         Add(
#             Sequential(
#                 Conv1d(K, K, 3, padding=1, bias=0),
#                 BatchNorm1d(K, track_running_stats=0),
#                 SiLU(1),
#                 Conv1d(K, K, 1, bias=0),
#             ),
#             Conv1d(K, K, 3, padding=1, bias=0),
#         ),
#         MaxPool1d(2),
#     ),    
#     Linear(K, K),
#     SiLU(1),
#     Linear(K, 7622),
# )
m = Sequential(
    EmbeddingBag(7622, K),
    ReLU(1),
    Linear(K, 7622),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .01, len(a)*100//512, pct_start=.01, div_factor=9, final_div_factor=999)
for t in tqdm(range(O.total_steps // 5000)):
    L = 0
    A = 0
    for t in range(10):
        for t in range(500):
            y = a[torch.randint(len(a),(512,))].long()
            x = m(y[:,:-1])
            y = y[:,-1]
            t = functional.cross_entropy(x, y)
            t.backward()
            o.step()
            o.zero_grad()
            O.step()
        L += t.detach()
        A += (x.argmax(1)==y).sum()
    print(L/10, A/len(y)/10)

In [ ]:
A/len(y)/10